# 17 Image generation

In [ ]:
import os
import sys
import pathlib
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt

os.environ["KERAS_BACKEND"] = "jax"

import keras

from IPython.display import clear_output
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

In [ ]:
def plot_history(history, metric):
    accuracy = history.history[metric]
    val_accuracy = history.history[f"val_{metric}"]
    epochs = range(1, len(accuracy) + 1)

    plt.plot(epochs, accuracy, "r--", label=f"Training {metric}")
    plt.plot(epochs, val_accuracy, "b", label=f"Validation {metric}")
    plt.title(f"Training and validation {metric}")
    plt.legend()
    plt.show()

In [ ]:
IMGGEN_DIR = pathlib.Path("image-generation")
IMGGEN_DIR.mkdir(exist_ok=True)

DATASET_DIR = pathlib.Path(IMGGEN_DIR / "datasets")
DATASET_DIR.mkdir(exist_ok=True)

MODELS_DIR = pathlib.Path("models")
MODELS_DIR.mkdir(exist_ok=True)

---

## Sampling from latent spaces of images

Many times we have seen that **information bottlenecks** can be a problem for us.

This time, we want to turn those to our advantage!

Imagine if we could compress images to a **lower-dimensional space**, a **latent space**, with good properties.

In such a space, each point (coordinate) would corresponds to a realistic image!

A *generator* or *decoder* module transforms a point in the latent space back to the source distribution, in our case: images.

Any point in the latent space can be sampled.

The sampling and reconstruction process means that new, never seen before, yet (hopefully) realistic, images can be created.

<!-- <img style="height:200px" src="images/chollet/figure17.1.png"> -->
<img style="height:200px" src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.1.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#sampling-from-latent-spaces-of-images), Figure 17.1


Advances in both Transformers and Diffusion models led to using **text embedding** to steer the generation process: this amounts to projecting text into the same latent space (turn them into tensors of a format that can be read by the model, and learnt from data), which then influence the generation.

<!-- <img style="height:200px" src="images/chollet/figure17.2.png"> -->
<img style="height:200px" src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.2.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#sampling-from-latent-spaces-of-images), Figure 17.2


There are a few main families of models for image generation:
- **Generative Adversarial Networks (GANs)**, not covered this year, but if interested you can look at [`lectures/09.more/2nd-ed.chapter12_part05_gans.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/2nd-ed.chapter12_part05_gans.ipynb)
- **Variational Autoencoders (VAEs)**
- **Diffusion Models** (and even combinations like the [Diffusion Transformer](https://github.com/milmor/diffusion-transformer-keras))

Note that even when a family of models like GANs is superseded by others (like Diffusion), usually key advances remain used in a variety of concept (in the case of GANs, the idea of an adversarial loss; Diffusion models integrate an heir of the VAE as their backbone, etc.).

---

## Variational Autoencoders

### Introduction

A **classic autoencoder** maps an image to a low dimensional fixed code (a vector) and then decodes back to the original space.

The autoencoder learns a kind of compression, and
The autoecoder then learns by comparing its decoded output with the input. It is **self-supervised**!

But these classic spaces are not structured in a useful way, *and* it's **deterministic**!

<!-- <img style="height:150px" src="images/chollet/figure17.3.png"> -->
<img style="height:150px"  src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.3.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#variational-autoencoders), Figure 17.3

**Variational autoencoders** do not compress data into a fixed code in the latent space, but learn the **parameters of a probability distribution** (its mean and variance).

The assumption is that the output image is the result of a statistical process.  
(The randomness of this process should be taken into account during encoding and decoding.)

The randomness (stochasticity) as well as the specific loss used improve robustness and forces the latent space to encode meaningful representations everywhere.

<!-- ![VAE](images/chollet/figure17.4.png) -->
<img src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.4.png">


[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#variational-autoencoders), Figure 17.4

#### Workflow

1. The **encoder** module maps an input image to a *mean* $\mu$ and a *variance* $\sigma$;
2. We **randomly sample**: $z \sim \mathcal{N}(\mu, \sigma^2)$ i.e. $z \sim \mu + \sigma \cdot \mathcal{N}(0, 1)$;
3. The **decoder** module maps $z$ back to the original image space.

#### Sampling from a normal distribution

Sampling from any Gaussian distribution:
$$
\Large
\bbox[5px,border:2px solid red]{
z \sim \mathcal{N}(\mu, \sigma^2)\\
}
$$
can be performed by sampling from a Standard Normal (mean 0, std 1) and shifting/scaling it:

$$
\Large
\bbox[5px,border:2px solid red]{
z \sim \underbrace{ \mu }_{ \text{shift} } +  \underbrace{ \sigma }_{ \text{scale} } \overbrace{ \mathcal{N}(0, 1) }^{\text{standard normal} }
}
$$

$z$: sample  
$\mu$: mean  
$\sigma^2$: variance, $\sigma$: standard deviation  
$\mathcal{N}$: normal distribution  

You can think of $\mu$ as a point in the latent space and $\sigma$ defining an area around this point.

The **regularisation loss** ensures that the $z$'s are clustered together at the centre of the latent space.

The stochastic sampling means that **slightly different latent vectors** will be generated **from the same source image**.

The decoder is attempting to **decode all the random latent vectors** emanating from the **same source** image to the **same target** image (identical to the source, we try to reconstruct it!).

=> **neighbouring points** in the latent space are decoded to the **same image**.

The $\sigma$ areas **overlap** so that a **continuous and structured representation** is built.

That representation is **densely packed**, so that **any sampled point** from it should yield a **new valid data point**.

New images are generated by decoding a selected point $z$ in the latent space.

See [this visualisation](https://www.youtube.com/watch?v=sV2FOdGqlX0).

#### The loss functions

We train our VAE with two losses:

1. A **reconstruction loss**: forces the decoded samples to match the initial inputs (our model predicts a value for each pixel, and since the pixels are values between 0 and 1 we can use the binary cross-entropy):
3. A **regularisation loss**: ensures the latent space is well-formed (in our case: is shaped like a standard normal Gaussian distribution), and helps reducing overfitting. The formula used in the model is actually the [Kullback-Leibler divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence), that measures the discrepancy between two distributions: in our case, the one the model outputs, and a standard normal distribution (the derivation for this is really quite involved, but if you're interested, you can have a look in the references notebook).

---

### Implementing a VAE with Keras

#### Encoder (from images to latent space)

In [ ]:
# todo: experiment with this
LATENT_DIM = 2

def build_encoder(latent_dim=2):
    image_inputs = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(image_inputs)
    x = keras.layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(16, activation="relu")(x)
    # MEAN
    z_mean = keras.layers.Dense(latent_dim, name="z_mean")(x)
    # LOG OF VARIANCE = log(sigma**2) (log for math/stability)
    z_log_var = keras.layers.Dense(latent_dim, name="z_log_var")(x)
    return keras.Model(image_inputs, [z_mean, z_log_var], name="encoder")
encoder = build_encoder(latent_dim=LATENT_DIM)

In [ ]:
encoder.summary()

#### Latent-space-sampling layer

In [ ]:
@keras.saving.register_keras_serializable()
class Sampler(keras.Layer):
    """From a mean mu and a log variance, sample a number with mu + sigma * epsilon"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # We need a seed generator to use functions from keras.random in call().
        self.seed_generator = keras.random.SeedGenerator()
        self.built = True

    def call(self, z_mean, z_log_var):
        batch_size = keras.ops.shape(z_mean)[0]
        z_size = keras.ops.shape(z_mean)[1]
        # draws a batch of random normal vectors
        epsilon = keras.random.normal(
            (batch_size, z_size), seed=self.seed_generator
        )
        # applies the VAE sampling formula (mu + sigma * epsilon)
        return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon

##### Examining our sampler

$$
\Large
e^{ \overbrace{ 0.5 * \log \sigma^2 }^{ \text{numerically stable} } } = e^{\log \sqrt{ \sigma^2 } } = \sqrt{ \sigma^2 } = \sigma
$$

In [ ]:
# Why the above formula works?

std = 1.4
var = std**2
log_var = np.log(var)
print(f"std: {std}")
print(f"var: {var}")
print(f"log var: {log_var}")
print(f"and back to std: {np.exp(0.5 * log_var)}")
print()
print("The log and the exp functions are reciprocal, & the log transforms exponentiation into multiplication:")
print("and 0.5 * log(var) == log(var ** 0.5) == log(sqrt(var)).")
print()
print("0.5 * log_var == np.log(np.sqrt(var))?", np.allclose(0.5 * log_var, np.log(np.sqrt(var))))

#### Decoder (from latent space to images)

In [ ]:
def build_decoder(latent_dim=2):
    latent_inputs = keras.Input(shape=(latent_dim,))
    x = keras.layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
    x = keras.layers.Reshape((7, 7, 64))(x)
    # CONV 2D TRANSPOSE
    x = keras.layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
    x = keras.layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
    decoder_outputs = keras.layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)
    return keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder = build_decoder(latent_dim=LATENT_DIM)

In [ ]:
decoder.summary()

##### Reminder: transposed convolutions

[`keras.layers.Conv2DTranspose`](https://www.tensorflow.org/api_docs/python/keras/layers/Conv2DTranspose) docs. See also [this article](https://towardsdatascience.com/types-of-convolutions-in-deep-learning-717013397f4d), and especially [that paper](https://arxiv.org/abs/1603.07285) ([github](https://github.com/vdumoulin/conv_arithmetic)).

<!-- ![Kaveh, stack overflow](images/deconv/deconv.1.png) -->
![Kaveh, stack overflow](https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/deconv/deconv.1.png?)

<small>Source: [Kaveh's answer to "In Keras what is the difference between Conv2DTranspose and Conv2D", stack overflow](https://stackoverflow.com/a/68980531)</small>

<!-- ![Kaveh, stack overflow](images/deconv/deconv.2.png) -->
![Kaveh, stack overflow](https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/deconv/deconv.2.png?)


<small>Source: [Kaveh's answer to "In Keras what is the difference between Conv2DTranspose and Conv2D", stack overflow](https://stackoverflow.com/a/68980531)</small>

#### The full VAE

In [ ]:
@keras.saving.register_keras_serializable()
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.sampler = Sampler()
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    def call(self, inputs):
        # GENERATION
        return self.encoder(inputs)

    def compute_loss(self, x, y, y_pred, sample_weight=None, training=True):
        # Argument x is the model's input.
        original = x
        # Argument y_pred is the output of call().
        z_mean, z_log_var = y_pred

        # This is our reconstructed image.
        reconstruction = self.decoder(self.sampler(z_mean, z_log_var))

        # LOSSES

        # a. reconstruction_loss: BCE between data & reconstructions (performed on each pixel,
        # then we sum across all pixels for each sample separately, then average
        reconstruction_loss = keras.ops.mean(
            keras.ops.sum(
                keras.losses.binary_crossentropy(x, reconstruction),
                axis=(1, 2)
            )
        )

        # b. KL divergence (regularisation loss)(→ the latent distribution is close to a Gaussian)
        kl_loss = -0.5 * (
            1 + z_log_var - keras.ops.square(z_mean) - keras.ops.exp(z_log_var)
        )

        # c. total
        total_loss = reconstruction_loss + keras.ops.mean(kl_loss)

        # d. update our metrics trackers
        # self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return total_loss

    # plumbing: making this model serializable/reloadable ---------------------------------------
    # https://claude.ai/share/97fd35a0-aca9-4bb5-83ac-96bc392528f7
    def build(self, input_shape):
        # The encoder and decoder are already built as functional models,
        # so we just need to mark this model as built.
        self.encoder.build(input_shape)
        self.built = True

    def get_config(self):
        # Serialize the encoder and decoder configs so the full model
        # can be reconstructed from its config alone (required for saving).
        base_config = super().get_config()
        return {
            **base_config,
            "encoder": keras.layers.serialize(self.encoder),
            "decoder": keras.layers.serialize(self.decoder),
        }

    @classmethod
    def from_config(cls, config):
        # Deserialize the encoder and decoder before passing to __init__.
        encoder = keras.layers.deserialize(config.pop("encoder"))
        decoder = keras.layers.deserialize(config.pop("decoder"))
        return cls(encoder=encoder, decoder=decoder, **config)

#### Dataset & Baseline

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
# mnist_digits = np.concatenate([x_train, x_test], axis=0)
# expand dim goes (B, W, H) -> (B, W, H, 1)
mnist_train = np.expand_dims(x_train, -1).astype("float32") / 255
mnist_val = np.expand_dims(x_test, -1).astype("float32") / 255

For a discussion of the baseline (not in the exam, if you're interested in studying the VAE in your CW, see the notebook [`lectures/10.more/chapter17_image-generation.VAE-metrics.ipynb`](https://github.com/jchwenger/blob/main/lectures/10.more/chapter17_image-generation.VAE-metrics.ipynb). The short answer is: use a classic dimensionality reduction technique like [Principal Components Analysis (PCA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) to create an Autoencoder (not variational) as a baseline. We can also use the same regularisation loss on the latent space, to see how far it is from a Gaussian.

#### Training

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(
    optimizer=keras.optimizers.Adam(),
    run_eagerly=True,
)

VAE_NAME = "vae-mnist.keras"

BATCH_SIZE = 128

history = vae.fit(
    mnist_train,
    validation_data=mnist_val,
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            filepath=MODELS_DIR / VAE_NAME,
            save_best_only=True,
        ),
    ],
)

In [ ]:
plot_history(history, "kl_loss")

In [ ]:
plot_history(history, "reconstruction_loss")

#### Use your trained model

In [ ]:
vae_reloaded = keras.models.load_model(MODELS_DIR / "vae-mnist.keras")

##### Encode and decode one image

In [ ]:
# select one MNIST image
n = np.random.randint(0, mnist_digits.shape[0])
img = mnist_digits[n:n+1]

# encode to mean and variance (each of dimension (batch, LATENT_DIM)
mu, logvar = vae_reloaded.encoder.predict(img, verbose=0)
# print(mu.shape, logvar.shape)

# sample a coordinate (LATENT_DIM) & reconstruct the image
z = Sampler()(mu, logvar)
reconstructed_img = vae_reloaded.decoder.predict(z, verbose=0)

fig, axs = plt.subplots(1,2)
axs[0].imshow(img[0], cmap="gray")
axs[0].set_title("Original image")
axs[0].axis("off")

axs[1].imshow(reconstructed_img[0], cmap="gray")
axs[1].set_title("Reconstructed image")
_ = axs[1].axis("off")

##### Generate an image from a random latent space coordinate

In [ ]:
# We can also just sample a coordinate in the latent space
z = keras.random.normal((1,LATENT_DIM))
x_decoded = vae_reloaded.decoder.predict(z, verbose=0)
plt.imshow(x_decoded.reshape((28,28)), cmap="gray")
plt.axis("off")

##### Interpolation in latent space: linear vs spherical interpolation

Given a latent space (and that can be this one, or the space of text embeddings in Diffusion models, see the example in the ["Exploring the latent space of a text-to-image model"](exploring-the-latent-space-of-a-text-to-image-model) section), we can experiment with the interpolation betwen points – gradually moving from one point to another in latent space, and generating outputs for each step (they should be valid outputs. 

Instead of a linear interpolation (`lerp`), it is recommended to use a special interpolation function called `slerp` to 'walk' between points in the latent space. This is short for **spherical linear interpolation**. That is because if the points in our latent space are distributed according to a multi-variate (more 2D or higher) Gaussian, it is advantageous not to cross the space in a straight line, but to walk along the perimeter of a circle/sphere (there are good reasons for this: [most points in a multivariate gaussian are located there](https://www.johndcook.com/blog/2011/09/01/multivariate-normal-shell/)). When we sample two random points in the latent space, they will very likely be somewhere on the surface of that sphere.  Thus, linearly interpolating between these two points would land us inside the sphere. We would no longer be on its surface. We would like to stay on the surface of the smooth manifold learned by our latent space — that’s where points have meaning for our decoder.

<!-- <img style="height:400px" src="images/chollet/figure17.16.png"> -->
<img style="height:400px" src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.16.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#sampling-from-latent-spaces-of-images), Figure 17.16


In [ ]:
def lerp(t, v1, v2):
    return (1 - t) * v1 + t * v2

def slerp(t, v1, v2):
    v1, v2 = keras.ops.cast(v1, "float32"), keras.ops.cast(v2, "float32")
    # normalize the vectors
    v1_norm = keras.ops.linalg.norm(keras.ops.ravel(v1))
    v2_norm = keras.ops.linalg.norm(keras.ops.ravel(v2))
    dot = keras.ops.sum(v1 * v2 / (v1_norm * v2_norm))
    # get the angle
    theta_0 = keras.ops.arccos(dot)
    # linear interpolation **on the angle**
    theta_t = theta_0 * t
    # use the angle to interpolate in coordinate space
    sin_theta_0 = keras.ops.sin(theta_0)
    sin_theta_t = keras.ops.sin(theta_t)
    s0 = keras.ops.sin(theta_0 - theta_t) / sin_theta_0
    s1 = sin_theta_t / sin_theta_0
    return s0 * v1 + s1 * v2

Demo of the linear vs spherical linear interpolation functions. There is a difference, if you test other seeds, although the difference is not 100% stark – it might be a good experiment to see if having a higher-dimensional latent space (than `2`, as now) makes the difference between the two methods clearer.

In [ ]:
keras.utils.set_random_seed(6)
zs = keras.random.normal((2,LATENT_DIM))
steps = 10

zs_lerp = []
for t in np.linspace(0, 1, steps):
    # linear interpolation
    zs_lerp.append(lerp(t, zs[0:1], zs[1:2]))

x_lerp = vae_reloaded.decoder.predict(keras.ops.concatenate(zs_lerp), verbose=0)

fig, axs = plt.subplots(1, steps, figsize=(12,5))
for i in range(steps):
    axs[i].imshow(x_lerp[i].reshape((28,28)), cmap="gray")
    axs[i].axis("off")

In [ ]:
zs_slerp = []
for t in np.linspace(0, 1, steps):
    # spherical linear interpolation    
    zs_slerp.append(slerp(t, zs[0:1], zs[1:2]))

x_slerp = vae_reloaded.decoder.predict(keras.ops.concatenate(zs_slerp), verbose=0)

fig, axs = plt.subplots(1, steps, figsize=(12,7))
for i in range(steps):
    axs[i].imshow(x_slerp[i].reshape((28,28)), cmap="gray")
    axs[i].axis("off")

##### Sampling a grid of images from the 2D latent space

In this example, we create a grid of points from `-1` to `1` for both `x` and `y`, create all pairs, decode all points from latent space back to images, and stitch them all into one picture. Note that we can use parallel computing to decode all points in one go (like a batch).

In [ ]:
def plot_latent_space(vae):
    if LATENT_DIM > 2:
        print(f"This plot only works for LATENT_DIM = 2 (currently {LATENT_DIM = }")
        print("Idea to expand it: interpolate in n steps between two samples,")
        print("Or use the same grid logic but using only two of the available latent dims.")
    n = 30
    img_size = 28
    # empty pixels
    figure = np.zeros((img_size * n, img_size * n))

    # linearly spaced values for x
    grid_x = np.linspace(-1, 1, n)
    # linearly spaced values for y
    grid_y = np.linspace(-1, 1, n)[::-1]

    # optimization, see: https://claude.ai/share/4fb657fb-42dd-4aec-80ab-9c3da07ef62a
    # 1. sample at all (xi, yi) coordinates
    z_samples = np.array([[xi, yi] for yi in grid_y for xi in grid_x])
    # 2. decode all images from samples in one batch
    x_decoded = vae.decoder.predict(z_samples, verbose=0)

    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            # 3. include image into our grid
            img = x_decoded[i * n + j].reshape(img_size, img_size)
            figure[
                i * img_size : (i + 1) * img_size,
                j * img_size : (j + 1) * img_size,
            ] = img

    plt.figure(figsize=(15, 15))
    start_range = img_size // 2
    end_range = n * img_size + start_range
    pixel_range = np.arange(start_range, end_range, img_size)
    sample_range_x = np.round(grid_x, 1)
    sample_range_y = np.round(grid_y, 1)
    plt.xticks(pixel_range, sample_range_x)
    plt.yticks(pixel_range, sample_range_y)
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.axis("off")
    plt.imshow(figure, cmap="Greys_r")    

In [ ]:
plot_latent_space(vae_reloaded)

## Diffusion models

### Introduction

**Diffusion** refers to the process of gradually adding noise to an image until the pixels are indistinguishable from noise (Gaussian noise, that has nice properties).

The process of transformation from image to noise (**forward**, whilst **reverse** is from noise to image, somewhat confusingly) is well-known and easy to compute. Given an image, we can produce a noisy version with any amount of noise added. Very useful for training, as we will see!

We can view this as a time axis, with 0 being the image and 1 being full noise. Going from image to noise is easy, however the *reverse* process is hard, and what we want to learn. By decomposing this into small steps, it so happens that models are able to learn this: we will train a model to **predict the noise to be removed** in a noisy image (and we can easily create infinite training data, so create both a noisy image at time `t`, and another at time `t-1`, which gives us the 'ground truth' noise that we compare with the model prediction).

<!-- <img style="height:250px" src="images/chollet/figure17.7.png"> -->
<img style="height:250px"  src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09/images/chollet/figure17.7.png">

[DLWP](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#diffusion-models), Figure 17.7

### The Oxford Flowers dataset

In [ ]:
fpath = keras.utils.get_file(
    origin="https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz",
    cache_dir=IMGGEN_DIR,
    cache_subdir=DATASET_DIR.name,    
    extract=True,
)

BATCH_SIZE = 32
IMAGE_SIZE = 128
IMAGES_DIR = os.path.join(fpath, "jpg")

dataset = keras.utils.image_dataset_from_directory(
    IMAGES_DIR,
    # We won't need the labels, just the images.
    labels=None,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    # Crops images when resizing them to preserve their aspect ratio
    crop_to_aspect_ratio=True,
)

dataset = dataset.rebatch(BATCH_SIZE, drop_remainder=True)
total_batches = dataset.reduce(0, lambda count, _: count + 1).numpy().item()
split = total_batches // 10
train_batches = total_batches - split
val_ds = dataset.take(split)
train_ds = dataset.skip(split)
print(f"{total_batches} batches.")

#### Test our dataset

In [ ]:
for batch in dataset:
    img = batch.numpy()[0]
    break
plt.imshow(img.astype("uint8"))

### A U-Net denoising autoencoder

The **U-Net**, originally developed for **semantic segmentation**, is an architecture that is widely used for implementing diffusion models but with some slight modifications:

<!-- ![u-net](images/diffusion/u-net.jpeg) -->
<img style="height: 400px;"   src="https://raw.githubusercontent.com/jchwenger/DLWP/main/lectures/09.more/images/diffusion/u-net.jpeg">

Source: [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)

The architecture has three stages:

1. A *downsampling stage*, made of several blocks of convolution layers, where the inputs get downsampled (for us, from their original 128 × 128 size down to a much smaller size, 16 × 16).
2. A *middle stage*, where the feature map has a constant size.
3. An *upsampling stage*, where the feature map get upsampled back to 128 × 128.


This architecture is directly inspired by autoencoders! Note that **unlike the VAE**, the U-Net does **not include a sampling step** in the middle!

In [ ]:
# Utility function to apply a block of layers with a residual connection
def residual_block(x, width):
    input_width = x.shape[3]
    if input_width == width:
        residual = x
    else:
        residual = keras.layers.Conv2D(width, 1)(x)
    x = keras.layers.BatchNormalization(center=False, scale=False)(x)
    x = keras.layers.Conv2D(width, 3, padding="same", activation="swish")(x)
    x = keras.layers.Conv2D(width, 3, padding="same")(x)
    x = x + residual
    return x

def get_model(image_size, widths, block_depth):
    noisy_images = keras.Input(shape=(image_size, image_size, 3))
    noise_rates = keras.Input(shape=(1, 1, 1))

    x = keras.layers.Conv2D(widths[0], 1)(noisy_images)
    n = keras.layers.UpSampling2D(image_size, interpolation="nearest")(noise_rates)
    x = keras.layers.Concatenate()([x, n])

    skips = []
    # Dowsampling stage
    for width in widths[:-1]:
        for _ in range(block_depth):
            x = residual_block(x, width)
            # Save the intermediate outputs for the Upsampling stage
            skips.append(x)
        x = keras.layers.AveragePooling2D(pool_size=2)(x)

    # Middle stage
    for _ in range(block_depth):
        x = residual_block(x, widths[-1])

    # Upsampling stage
    for width in reversed(widths[:-1]):
        x = keras.layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            # Reuse the intermediate Downsampling outputs
            x = keras.layers.Concatenate()([x, skips.pop()])
            x = residual_block(x, width)

    # We set the kernel initializer for the last layer to "zeros,"
    # making the model predict only zeros after initialization (that
    # is, our default assumption before training is "no noise").
    pred_noise_masks = keras.layers.Conv2D(3, 1, kernel_initializer="zeros")(x)

    # Creates the functional model
    return keras.Model([noisy_images, noise_rates], pred_noise_masks)

### The concepts of diffusion time and diffusion schedule

The diffusion process is a **series of steps** in which we apply our denoising U-Net to **remove a small amount of noise from an image**, starting with a pure-noise image, and ending with a pure-signal image.

The key idea is that we know how to **apply any amount of noise** to an image, a priori, without the need of a neural net. That way, we can create as many training samples we need: we can compute the noise at times $t$ and $t-1$, get feed the noisy image at time $t$ into the model, and use the pre-computed noise at time $t-1$ as the ground truth to make the model learn how to produce the right amount of noise to remove at that step.

There are many types of schedules, an active area of research! This one preserves the relationship `noise_rates ** 2 + signal_rates ** 2 == 1`. A good post about schedules can be found [here](https://sander.ai/2024/06/14/noise-schedules.html).

In [ ]:
def diffusion_schedule(
    diffusion_times,
    min_signal_rate=0.02,
    max_signal_rate=0.95,
):
    start_angle = keras.ops.cast(keras.ops.arccos(max_signal_rate), "float32")
    end_angle = keras.ops.cast(keras.ops.arccos(min_signal_rate), "float32")
    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)
    signal_rates = keras.ops.cos(diffusion_angles)
    noise_rates = keras.ops.sin(diffusion_angles)
    return noise_rates, signal_rates

#### Test our schedule

In [ ]:
# 0 → 1 diffusion times
diffusion_times = np.linspace(0, 1, 100)

noise_rates, signal_rates = diffusion_schedule(diffusion_times, min_signal_rate=0, max_signal_rate=1)

diffusion_times = keras.ops.convert_to_numpy(diffusion_times)
noise_rates = keras.ops.convert_to_numpy(noise_rates)
signal_rates = keras.ops.convert_to_numpy(signal_rates)

plt.figure(figsize=(5,5))
plt.plot(noise_rates, signal_rates, label=r"$\text{noise\_rates}^2 + \text{signal\_rates}^2 = 1$")
plt.title("Cosine Relationship between Noise and Signal Rates")
plt.xlabel("Noise Rates")
plt.ylabel("Signal Rates")
plt.xlim(0, 1.1)
plt.ylim(0, 1.1)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(diffusion_times, noise_rates, label="Noise rate")
plt.plot(diffusion_times, signal_rates, label="Signal rate")

plt.title("Diffusion time\n(at $t = 0.0$, no noise, full signal; vice versa at $t = 1.0$)")
plt.grid()
_ = plt.legend()

### The Diffusion Model

The things to note:

- the *loss function* (`self.loss` and `compute_loss`): we use the `MeanAbsoluteError` at a pixel level between the real (computed) noise and the predicted noise: `mean(abs(real_noise_mask - predicted_noise_mask))`.
- A *normalizer* — the noise we’ll add to the images will have unit variance and zero mean, so we’d like our images to be normalized in the same way.

The **training process** will be as follows. Given a batch of clean images, we will:

- **Normalize** the images.
- Sample **random diffusion times** (as said, it is straightforward to compute what noise to **add** at any time).
- Compute corresponding **noise rates** and **signal rates** (using the diffusion schedule).
- Add **the noise to the clean images** (based on the computed noise rates and signal rates), thus creating training samples on the spot.
- Feed the noisy images to the model, which will **predict the noise to be removed**.
- Compare the predicted noise with the ground truth noise (that we compute on the spot as well).

The **generation process**, on the other hand, is:

- Start with some **random (Gaussian) noise** (a fully noised 'image').
- For `diffusion_steps`, do:
    - Run the noisy image through the model to get the predicted noise to be removed.
    - Use that predicted noise (like a mask) gradually to remove the noise from the image (note that in our implementation the removal happens through a weighted sum: `(noisy_images - noise_rates * pred_noise_masks) / signal_rates`, which gradually goes from more noise to more signal.

In [ ]:
@keras.saving.register_keras_serializable()
class DiffusionModel(keras.Model):
    def __init__(self, image_size, widths, block_depth, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.widths = widths
        self.block_depth = block_depth
        
        self.denoising_model = get_model(image_size, widths, block_depth)
        self.seed_generator = keras.random.SeedGenerator()
        # Our loss function
        self.loss = keras.losses.MeanAbsoluteError()
        # We'll use this to normalize input images.
        self.normalizer = keras.layers.Normalization()

    def denoise(self, noisy_images, noise_rates, signal_rates):
        # The denoising model predicts the noise to be removed
        pred_noise_masks = self.denoising_model([noisy_images, noise_rates])
        # Reconstructs the predicted clean image (weighted sum with a bit less noise)
        # (noisy_images = pred_image * signal_rates + noise_rates * pred_noise_masks
        # but solving for pred_image), see: https://claude.ai/share/a935bbd6-aa03-4b86-883f-8099a900e196
        pred_images = (
            noisy_images - noise_rates * pred_noise_masks
        ) / signal_rates
        return pred_images, pred_noise_masks

    def call(self, images):
        images = self.normalizer(images)
        # Samples random noise masks
        noise_masks = keras.random.normal(
            (BATCH_SIZE, self.image_size, self.image_size, 3),
            seed=self.seed_generator,
        )
        # Samples random diffusion times (between zero and one)
        diffusion_times = keras.random.uniform(
            (BATCH_SIZE, 1, 1, 1),
            minval=0.0,
            maxval=1.0,
            seed=self.seed_generator,
        )
        noise_rates, signal_rates = diffusion_schedule(diffusion_times)
        # Create noisy images by mixing noise and image (like a weighted sum)
        noisy_images = signal_rates * images + noise_rates * noise_masks
        # Denoises them
        pred_images, pred_noise_masks = self.denoise(
            noisy_images, noise_rates, signal_rates
        )
        return pred_images, pred_noise_masks, noise_masks

    def compute_loss(self, x, y, y_pred, sample_weight=None, training=True):
        _, pred_noise_masks, noise_masks = y_pred
        return self.loss(noise_masks, pred_noise_masks)

    def generate(self, num_images, diffusion_steps):
        # Starts from Gaussian noise
        noisy_images = keras.random.normal(
            (num_images, self.image_size, self.image_size, 3),
            seed=self.seed_generator,
        )
        step_size = 1.0 / diffusion_steps
        for step in range(diffusion_steps):
            # Determine which timestep we're at
            diffusion_times = keras.ops.ones((num_images, 1, 1, 1)) - step * step_size
            # Computes appropriate noise and signal rates
            noise_rates, signal_rates = diffusion_schedule(diffusion_times)
            # Calls denoising model
            pred_images, pred_noises = self.denoise(
                noisy_images, noise_rates, signal_rates
            )
            # Computes noise and signal rates for the next timestep
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = diffusion_schedule(
                next_diffusion_times
            )
            # Update the image by mixing image and noise (gradually we remove the noise)
            noisy_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )
        # Denormalizes images so their values fit between 0 and 255
        images = (
            self.normalizer.mean + pred_images * self.normalizer.variance**0.5
        )
        return keras.ops.clip(images, 0.0, 255.0)

    # plumbing: making this model serializable/reloadable ---------------------------------------
    # https://chatgpt.com/share/69bd2782-4a00-8005-a7d9-e9e0e052dd88
    def build(self, input_shape):
        super().build(input_shape)
        
    def get_config(self):
        config = super().get_config()
        config.update({
            "image_size": self.image_size,
            "widths": self.widths,
            "block_depth": self.block_depth,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)        

#### Visualisation Callback

A nice callback to see images being generated during training.

In [ ]:
class VisualizationCallback(keras.callbacks.Callback):
    def __init__(self, diffusion_steps=20, num_rows=3, num_cols=6, save_every=10):
        self.diffusion_steps = diffusion_steps
        self.num_rows = num_rows
        self.num_cols = num_cols
        self.save_every = save_every

    def on_epoch_end(self, epoch=None, logs=None):
        if epoch % self.save_every != 0:
            return
        # generate images
        generated_images = self.model.generate(
            num_images=self.num_rows * self.num_cols,
            diffusion_steps=self.diffusion_steps,
        )
        # plot them
        plt.figure(figsize=(self.num_cols * 2.0, self.num_rows * 2.0))
        for row in range(self.num_rows):
            for col in range(self.num_cols):
                i = row * self.num_cols + col
                plt.subplot(self.num_rows, self.num_cols, i + 1)
                img = generated_images[i].numpy().astype("uint8")
                plt.imshow(img)
                plt.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close()

### Training

#### Learning Rate Schedule

In [ ]:
# https://keras.io/api/optimizers/learning_rate_schedules/inverse_time_decay/
schedule = keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.1,
)

##### Test our schedule

In [ ]:
x = range(0, 10000, 100)
y = [keras.ops.convert_to_numpy(schedule(step)) for step in x]
plt.plot(x, y)
plt.xlabel("Train Step")
plt.ylabel("Learning Rate")
plt.show()

#### Compilation

In [ ]:
model = DiffusionModel(IMAGE_SIZE, widths=[32, 64, 96, 128], block_depth=2)
# Computes the mean and variance necessary to perform normalization —
# don't forget it!
model.normalizer.adapt(dataset)

model.compile(
    optimizer=keras.optimizers.AdamW(
        # Configures the learning rate decay schedule
        learning_rate=schedule,
        # Turns on Polyak averaging
        use_ema=True,
        # Configures how often to overwrite the model's weights with
        # their exponential moving average
        ema_overwrite_frequency=100,
    ),
)

DIFF_NAME = "diffusion-oxford-flowers.keras"

#### Training

In [ ]:
history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs=100,
    callbacks=[
        VisualizationCallback(),
        keras.callbacks.ModelCheckpoint(
            filepath=MODELS_DIR / DIFF_NAME,
            save_best_only=True,
        ),
    ],
    steps_per_epoch = train_batches,
)

### Generating images with our trained model

In [ ]:
model_reloaded = keras.models.load_model(MODELS_DIR / DIFF_NAME)
model_reloaded.normalizer.adapt(dataset)

In [ ]:
generated_images = model_reloaded.generate(num_images=1, diffusion_steps=20)
img = generated_images[0].numpy().astype("uint8")
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()
plt.close()

## Save models to Google Drive


In [ ]:
EXPORT=False

if EXPORT:
    # zip models
    !zip vae-diffusion.models.zip {MODELS_DIR}/*
    # connect to drive
    from google.colab import drive
    drive.mount('/content/drive')
    # copy zip to drive (adjust folder as needed)
    !cp vae-diffusion.models.zip drive/MyDrive/gold/IS53024B-Artificial-Intelligence/models

---

## Text-to-image models

Modern diffusion models have more interesting characteristics:

- They allow you to steer the diffusion process using **text embeddings** (a certain form of **conditioning** of our U-Net, which ended up also called **prompting**, like with LLMs).
- That process being, yet again, akin to moving in a certain direction in the latent space (of all possible images), the same text can be a **positive prompt** (move 'in that direction' in the latent space) or a **negative** one (move 'in the opposite direction').
- The first implementations of this steering process used a classifier (that was **classifier guidance**), but by being clever with algebra it is possible to use only the difference between the conditioned and the unconditioned outputs, without any classifier (**classifier-free guidance**).
- It is possible to experiment with interpolation: the example in the book is between two text embeddings – in the same way as with VAEs, it is recommended to use **spherical linear interpolation** (`slerp`) rather than plain `lerp`.

Read more about it [in the book](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#text-to-image-models).

---

## Wrapping up

- **Encoding**: we learn a **latent spaces** capturing statistical information about a dataset of images.
- **Decoding**: new images are produced by sampling and decoding points from the latent space.
- The latent space **compresses** the data in a way that is not dissimilar to **word embeddings**.

Three major approaches to do this: **Variational Autoencoders** (VAEs), **Diffusion Models** and **Generative Adversarial Networks** (GANs).

### VAEs

- **VAEs** result in **highly structured, continuous latent representations**.
- **Classical Autoencoders** learn a **deterministic** encoding/decoding pipeline, whereas **Variational Autoencoders** include a **sampling/stochastic step** between encoder and decoder.
- They work well for **image editing in latent space**: face swapping, turning a frowning face into a smiling face, and so on.
- VAEs also work nicely for **latent-space-based animations** e.g. a walk along a cross section of the latent space, showing a starting image slowly morphing into different images in a continuous way.

### Diffusion

- **Diffusion models** learn to **predict the noise** to be removed in a noisy image.
- The big assumption is that we can learn to decompose a hard problem (going from full noise to image) into small steps (remove a bit of noise).
- During the training process, we can easily and precisely **compute the amount of noise to add** at any step, without the need of a model. This allows us to create an arbitrary number of training samples (given the noisy images at time `t` and `t-1`, that we can easily create, we can train the model to predict the noise to be removed to go from `t` to `t-1`, for any `t`).
- Our trajectory from noise to image (and back) is viewed as **time**, and a **noise schedule** controls **how much noise and signal** are present in the image at any step.
- During **inference**, we start from pure (Gaussian) noise, and iteratively get the model to predict the noise, that we remove, before predicting again.
- The **U-Net**, the model trained to predict the noise, is an **autoencoder**, but **without a sampling/stochastic step** in the middle (and with residual connections between encoder and decoder).

### Further experiments

- Train on other datasets. For instance, you could try the **Celeb Faces Attributes** (CelebA) dataset, or FashionMNIST, instead of MNIST. It’s a free-to-download image dataset containing more than 200,000 celebrity portraits. Available in the external [TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/celeb_a) module, [Kaggle](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset) and the [original project page](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html). See the GAN notebook!
- **Interpolation**: one way to visualise a 'path' between two images is to encode each image to get `z_1` and `z_2`, their respective latent vectors. Then use function to create a series of [interpolated vectors between the two](https://github.com/soumith/dcgan.torch/issues/14#issuecomment-525160139): then, decode all these using the decoder, and plot the results, which should be a smooth-ish transition between the two images, with each intermediate image still being close to the dataset!
- **Feature vectors**: in order to isolate and work with a **feature vector**, the strategy is as follows:
  - gather as many examples of images with, and without, the desired feature, as possible;
  - for each of those, use the encoder to generate a vector `z`;
  - average the values of all vectors with and without the feature into two vectors: `avg_z_with`, `avg_z_without`;
  - to get the vector that moves from 'without' to 'with' the feature, compute: `z_feature = avg_z_with - avg_z_without`;
  - now you should be able to edit an image, by encoding it using the encoder, into `z_img`, then adding the feature vector to it (you can use some scaling to vary the strength of the modification): `z_img_modified = z_img + scale_factor * z_feature`;
  - finally, use the decoder to generate an image from `z_img_modified`.
- Have a look at the 'extract_attrib_vector' function [here](https://github.com/eduhrami/Hands-On-Image-Generation-with-TensorFlow-2.0/blob/master/Chapter02/ch2_vae_faces.ipynb) (it uses label logic from CelebA, but you could select the images your self). See also [this notebook](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/03_vae/03_vae_faces/vae_faces.ipynb) and its [utils](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/03_vae/03_vae_faces/vae_utils.py).